# voidface training on Kaggle (P100, 12h sessions, resumable)

Trains `voidface.generator.architecture.Voidface` against the ensemble in `samples/configs/train_kaggle_p100.toml` (fallback: `train_full.toml`).

Session shape:
- Target 10h wall-clock (2h safety margin under Kaggle's 12h cap)
- Checkpoints every 500 steps to `/kaggle/working/checkpoints/`
- Resumable across sessions via a Kaggle Dataset (`voidface-training-ckpts`).

**Setup before Run All**
1. Right sidebar -> **Accelerator: GPU P100**
2. **Internet: On** (needs phone-verified Kaggle account)
3. **+ Add Input** -> FFHQ dataset (e.g. `chelove4draste/ffhq-512x512` or `denislukovnikov/ffhq256-images-only`).
4. After the first run, **+ Add Input** -> your own `voidface-training-ckpts` dataset to resume.

**Secrets:** none. Kaggle notebooks are auto-authenticated for `kaggle datasets version`. Do NOT paste `kaggle.json` contents here - this notebook is public if you make it public.

In [ ]:
# Cell 1 - environment info
import sys, time, platform

print(f"Python:    {sys.version.split()[0]}")
print(f"Platform:  {platform.platform()}")
print(f"Time UTC:  {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}")

try:
    import torch
    print(f"torch:     {torch.__version__}")
    print(f"CUDA:      {torch.version.cuda}")
    print(f"CUDA avail:{torch.cuda.is_available()}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU:       {torch.cuda.get_device_name(0)}")
        print(f"GPU RAM:   {props.total_memory / 1e9:.1f} GB")
        print(f"Compute:   {props.major}.{props.minor}")
        assert 'P100' in torch.cuda.get_device_name(0) or 'T4' in torch.cuda.get_device_name(0) or 'V100' in torch.cuda.get_device_name(0), \
            f"Unexpected accelerator {torch.cuda.get_device_name(0)!r}. Sidebar -> Accelerator -> GPU P100."
    else:
        raise RuntimeError('No CUDA device. Sidebar -> Accelerator -> GPU P100.')
except ImportError:
    print('torch not installed; cell 2 will install it.')

In [ ]:
# Cell 2 - install voidface's minimal training deps.
#
# Kaggle GPU images already ship torch/torchvision/opencv/pillow/numpy;
# we only add what voidface needs on top. `uv sync --frozen` isn't a
# fit here because Kaggle's base image would fight the lockfile
# platform markers. Pin loose upper bounds, quiet pip.
import subprocess, sys

PIP = [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-warn-conflicts', '--upgrade-strategy', 'only-if-needed']

EXTRAS = [
    'diffusers>=0.27',
    'transformers>=4.40',
    'huggingface_hub>=0.24',
    'lpips>=0.1.4',
    'pytorch-msssim>=1.0',
    'einops>=0.7',
    'structlog>=24.1',
    'tomli>=2.0',
    'opencv-python-headless>=4.9',
    # GFPGAN + its face-restoration deps for the bilevel restorer.
    'facexlib>=0.3',
    'gfpgan>=1.3.8',
    'basicsr>=1.4.2',
    # Publishing checkpoints back to a Kaggle Dataset.
    'kaggle>=1.6',
]
subprocess.check_call(PIP + EXTRAS)
print('dependencies installed')

In [ ]:
# Cell 3 - clone voidface and pin to a specific commit.
#
# Bump VOIDFACE_COMMIT when you want a newer training loop.
import os, subprocess

VOIDFACE_COMMIT = 'b2b07c5998e1568ac2dd445fd29768d9c15e4933'  # HEAD at notebook authoring time
REPO_URL = 'https://github.com/JakubZitko/voidface.git'
REPO_DIR = '/kaggle/working/voidface'

os.chdir('/kaggle/working')
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--quiet', REPO_URL, REPO_DIR])
subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--quiet', 'origin'])
subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', '--quiet', VOIDFACE_COMMIT])
head = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip()
print(f'HEAD: {head}')
assert head.startswith(VOIDFACE_COMMIT[:12]), 'checkout drifted from pinned commit'

# Editable installs so we can import voidface.* and voidface_cli.*.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{REPO_DIR}/tools/cli'])

# Sanity import.
sys.path.insert(0, f'{REPO_DIR}/src')
sys.path.insert(0, f'{REPO_DIR}/tools/cli')
import voidface, voidface_cli  # noqa: F401
print(f'voidface package OK, version {getattr(voidface, "__version__", "unknown")}')

In [ ]:
# Cell 4 - locate the attached FFHQ dataset.
#
# Kaggle mounts each attached dataset at /kaggle/input/<slug>/. We scan
# for the first mount that looks like a folder of face images.
import os, glob
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists():
    raise SystemExit('No /kaggle/input. Attach a dataset via + Add Input.')

print('/kaggle/input contents:')
for entry in sorted(INPUT_ROOT.iterdir()):
    print(f'  - {entry.name}')

# Preferred order: explicit ffhq mounts, then anything with image files.
CANDIDATE_SLUGS = [
    'ffhq', 'ffhq-512x512', 'ffhq256-images-only', 'ffhq-256x256',
    'flickrfaceshq-dataset-ffhq', 'ffhq-face-data-set', 'flickr-faces-hq-dataset-ffhq',
    'ffhq-224k', 'celebahq-resized-256x256',
]

def _looks_like_faces(p: Path) -> bool:
    # Count image files up to a small cap (Kaggle mounts are fast).
    hits = 0
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG'):
        for _ in p.rglob(ext):
            hits += 1
            if hits >= 32:
                return True
    return hits >= 32

FFHQ_DIR = None
for slug in CANDIDATE_SLUGS:
    candidate = INPUT_ROOT / slug
    if candidate.exists():
        FFHQ_DIR = candidate
        break

if FFHQ_DIR is None:
    for candidate in sorted(INPUT_ROOT.iterdir()):
        if candidate.is_dir() and 'ckpt' not in candidate.name and 'checkpoint' not in candidate.name:
            if _looks_like_faces(candidate):
                FFHQ_DIR = candidate
                break

if FFHQ_DIR is None:
    raise SystemExit(
        'No FFHQ-like image folder found under /kaggle/input.\n'
        'Attach e.g. chelove4draste/ffhq-512x512 or denislukovnikov/ffhq256-images-only via + Add Input.'
    )

# If the dataset ships images in a subfolder (common), descend one level.
if not any(FFHQ_DIR.glob('*.png')) and not any(FFHQ_DIR.glob('*.jpg')):
    subdirs = [p for p in FFHQ_DIR.iterdir() if p.is_dir()]
    for sub in subdirs:
        if _looks_like_faces(sub):
            FFHQ_DIR = sub
            break

n_images = sum(1 for _ in FFHQ_DIR.rglob('*.png')) + sum(1 for _ in FFHQ_DIR.rglob('*.jpg'))
print(f'\nFFHQ_DIR = {FFHQ_DIR}  ({n_images} image files found)')
assert n_images > 100, f'Only {n_images} images at {FFHQ_DIR}; check the dataset attachment.'

In [ ]:
# Cell 5 - load the training config and override the paths for Kaggle.
#
# Preferred config is samples/configs/train_kaggle_p100.toml (P100-tuned:
# batch 2, res 384, grad-ckpt on). Falls back to train_full.toml with
# manual downgrades if the Kaggle-specific config isn't in the pinned
# commit yet.
try:
    import tomllib  # py>=3.11 stdlib
except ImportError:  # py3.10 fallback (cell 2 installs tomli)
    import tomli as tomllib
from pathlib import Path

CFG_PRIMARY = Path(REPO_DIR) / 'samples/configs/train_kaggle_p100.toml'
CFG_FALLBACK = Path(REPO_DIR) / 'samples/configs/train_full.toml'

if CFG_PRIMARY.exists():
    CFG_PATH = CFG_PRIMARY
    print(f'using {CFG_PATH}')
else:
    CFG_PATH = CFG_FALLBACK
    print(f'{CFG_PRIMARY.name} not present at this commit; falling back to {CFG_PATH.name}')

with open(CFG_PATH, 'rb') as f:
    cfg = tomllib.load(f)

cfg.setdefault('data', {})
cfg.setdefault('experiment', {})

# Overrides for Kaggle P100 - safe defaults if config lacks them.
cfg['data']['directory'] = str(FFHQ_DIR)
cfg['data'].setdefault('resolution', 384)
cfg['data'].setdefault('batch_size', 2)
cfg['data'].setdefault('augment', True)

CKPT_LIVE = Path('/kaggle/working/checkpoints')
CKPT_LIVE.mkdir(parents=True, exist_ok=True)
cfg['experiment']['checkpoint_dir'] = str(CKPT_LIVE)
cfg['experiment']['checkpoint_every'] = 500
cfg['experiment'].setdefault('steps', 25_000)
cfg['experiment'].setdefault('log_every', 50)
cfg['experiment'].setdefault('seed', 0)

print('resolved config:')
for section, body in cfg.items():
    if isinstance(body, dict):
        print(f'  [{section}]')
        for k, v in body.items():
            print(f'    {k} = {v!r}')

In [ ]:
# Cell 6 - detect the latest checkpoint from any prior session.
#
# Search order:
#   1. /kaggle/working/checkpoints/       (this session; empty on first run)
#   2. /kaggle/input/voidface-training-ckpts/  (dataset attached from prior session)
#   3. /kaggle/input/<anything with 'ckpt' or 'checkpoint' in the name>
import glob, os, re
from pathlib import Path

def _find_ckpts(root: Path):
    if not root.exists():
        return []
    return sorted(root.rglob('voidface-step-*.pt')) + sorted(root.rglob('ckpt_*.pt'))

SEARCH_ROOTS = [CKPT_LIVE]
for entry in Path('/kaggle/input').iterdir() if Path('/kaggle/input').exists() else []:
    if entry.is_dir() and ('ckpt' in entry.name.lower() or 'checkpoint' in entry.name.lower() or 'voidface' in entry.name.lower()):
        SEARCH_ROOTS.append(entry)

def _step_of(p: Path) -> int:
    m = re.search(r'step-(\d+)', p.name) or re.search(r'ckpt_(\d+)', p.name)
    return int(m.group(1)) if m else -1

all_ckpts = []
for root in SEARCH_ROOTS:
    for p in _find_ckpts(root):
        all_ckpts.append(p)

if all_ckpts:
    # Prefer highest step; break ties by mtime.
    all_ckpts.sort(key=lambda p: (_step_of(p), p.stat().st_mtime))
    RESUME_PATH = all_ckpts[-1]
    RESUME_STEP = _step_of(RESUME_PATH)
    print(f'[resume] latest checkpoint: {RESUME_PATH}')
    print(f'[resume] resuming at step {RESUME_STEP}')
else:
    RESUME_PATH = None
    RESUME_STEP = 0
    print('[resume] no prior checkpoint - starting from scratch')

In [ ]:
# Cell 7 - run training.
#
# We call voidface.core.train.train_generator directly (not via CLI
# subprocess) so the notebook can inspect the returned history object
# and cut a loss curve at the end. This cell mirrors the CLI's setup
# (tools/cli/voidface_cli/commands/train.py) with an added time budget:
# we stop cleanly at TIME_BUDGET_SEC even if steps remain, so the
# session ends before Kaggle's 12h wall kills us mid-write.
import os, time
from pathlib import Path

# ------- memory / fragmentation env before torch touches CUDA ---------
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
from torch.utils.data import DataLoader

from voidface.core.eot import EotConfig, EotSampler
from voidface.core.loss import (
    CompositeLoss, LossWeights,
    arcface_identity_loss, retinaface_suppression_loss, vae_gray_latent_loss,
)
from voidface.core.train import TrainConfig, train_generator
from voidface.data.datasets import FolderImageDataset
from voidface.eval.perceptual import load_lpips
from voidface.generator.architecture import Voidface, VoidfaceConfig
from voidface.models.restorers.identity import IdentityRestorer
from voidface.models.restorers.sampler import RestorerSampler, SamplerConfig
from voidface.util.log import configure_logging, get_logger
from voidface_cli.common import renormalize_weights

configure_logging(level='INFO')
log = get_logger('voidface.kaggle.train')
device = torch.device('cuda')

# ------------------------- dataset + loader --------------------------
data_conf = cfg['data']
resolution = int(data_conf['resolution'])
batch_size = int(data_conf['batch_size'])
dataset = FolderImageDataset(
    Path(data_conf['directory']).expanduser(),
    resolution=resolution,
    augment=bool(data_conf.get('augment', True)),
)
loader = DataLoader(
    dataset, batch_size=batch_size, shuffle=True,
    num_workers=2, persistent_workers=True, pin_memory=True,
)
log.info('dataset.loaded', size=len(dataset), resolution=resolution, batch=batch_size)

# ------------------------- generator + resume ------------------------
optim_conf = cfg.get('optim', {})
generator = Voidface(
    VoidfaceConfig(epsilon=float(optim_conf.get('epsilon_frac', 12.0 / 255.0)))
)
if RESUME_PATH is not None:
    log.info('generator.resume', path=str(RESUME_PATH))
    payload = torch.load(RESUME_PATH, map_location='cpu', weights_only=False)
    state_dict = payload['state_dict'] if isinstance(payload, dict) and 'state_dict' in payload else payload
    generator.load_state_dict(state_dict)
    log.info('generator.resumed', from_step=payload.get('step') if isinstance(payload, dict) else None)
log.info('generator.built', params=sum(p.numel() for p in generator.parameters()))

# ---------------- targets (build the same way the CLI does) ----------
targets_conf = cfg.get('targets', {})
target_losses, target_static_data, weights_targets = {}, {}, {}
vae = None

def _w(name, default):
    return float(targets_conf.get(name, {}).get('weight', default))

if targets_conf.get('detector', {}).get('enabled', True) or 'retinaface' in targets_conf:
    from voidface.models.detectors.retinaface import RetinaFace
    detector = RetinaFace(device=device)
    target_losses['detector'] = (detector, retinaface_suppression_loss)
    weights_targets['detector'] = _w('detector', 0.35)

if targets_conf.get('recognizer', {}).get('enabled', True) or 'arcface' in targets_conf:
    from voidface.models.recognizers.arcface import Arcface
    recognizer = Arcface(device=device)
    target_losses['recognizer'] = (recognizer, arcface_identity_loss)
    weights_targets['recognizer'] = _w('recognizer', 0.40)

if targets_conf.get('vae', {}).get('enabled', False) or 'sd15_vae' in targets_conf:
    from voidface.models.vaes.sd15 import Sd15Vae
    vae = Sd15Vae(device=device)
    gray = vae.encode_gray_target(height=resolution, width=resolution)
    target_losses['vae'] = (vae, vae_gray_latent_loss)
    target_static_data['vae'] = gray
    weights_targets['vae'] = _w('vae', 0.20)

renormalize_weights(weights_targets)
log.info('targets.built', targets=sorted(weights_targets), weights=weights_targets)

# ---------------------------- composite loss -------------------------
percep_conf = cfg.get('loss', {}).get('perceptual', {})
loss_conf = cfg.get('loss', {})
lpips_weight = float(percep_conf.get('lpips_weight', 0.10))
lpips_fn = load_lpips(net='alex', device=device) if lpips_weight > 0 else None
weights = LossWeights(
    targets=weights_targets,
    lpips=lpips_weight,
    total_variation=float(percep_conf.get('tv_weight', 0.01)),
    bilevel_lpips=float(loss_conf.get('bilevel_lpips', 0.0)),
    normalize_per_target=bool(loss_conf.get('normalize_per_target', False)),
    normalization_ema_decay=float(loss_conf.get('normalization_ema_decay', 0.99)),
)
composite = CompositeLoss(
    weights=weights, target_losses=target_losses,
    target_static_data=target_static_data, lpips=lpips_fn,
)

# --------------------------------- EOT -------------------------------
eot_conf = cfg.get('eot', {})
eot = EotSampler(EotConfig(
    samples=int(eot_conf.get('k', optim_conf.get('eot_samples', 2))),
    resize_factors=tuple(eot_conf.get('resize_factors', (0.75, 1.0, 1.5))),
    gaussian_sigma=tuple(eot_conf.get('gaussian_sigma', (0.0, 0.5, 1.0))),
    jpeg_qualities=tuple(int(q) for q in eot_conf.get('jpeg_qualities', ())),
    seed=int(cfg['experiment'].get('seed', 0)),
))

# ---------------------------- restorer mix ---------------------------
restorers_conf = cfg.get('restorers', {}) or cfg.get('bilevel', {}).get('restorer_sample_probability', {})
restorer_options = []
for name, weight in (restorers_conf or {'identity': 1.0}).items():
    w = float(weight)
    if w <= 0.0:
        continue
    if name == 'identity':
        restorer_options.append((IdentityRestorer(), w))
    elif name in ('sd15-vae', 'sd15_vae'):
        from voidface.models.restorers.sd_vae import Sd15VaeRestorer
        assert vae is not None, 'sd15-vae restorer needs [targets.vae] enabled'
        restorer_options.append((Sd15VaeRestorer(encoder=vae), w))
    elif name == 'gfpgan':
        from voidface.models.restorers.gfpgan import GfpganRestorer
        det = target_losses.get('detector', (None,))[0]
        if det is None:
            from voidface.models.detectors.retinaface import RetinaFace
            det = RetinaFace(device=device)
        restorer_options.append((GfpganRestorer(detector=det, device=device), w))
    else:
        log.warn('restorer.unknown', name=name)
if not restorer_options:
    restorer_options.append((IdentityRestorer(), 1.0))
restorer_sampler = RestorerSampler(restorer_options, SamplerConfig(seed=0))
log.info('restorers.built', mix=restorer_sampler.probabilities())

# --------------- time-budget wrapper: stop before 10h ---------------
TIME_BUDGET_SEC = 10 * 60 * 60  # 10h, 2h under Kaggle's 12h cap
start_wall = time.monotonic()

class _TimeBudgetLoader:
    """Wraps a DataLoader; stops iteration once the wall budget is spent."""
    def __init__(self, inner, budget_sec):
        self._inner = inner
        self._budget = budget_sec
    def __iter__(self):
        for batch in self._inner:
            if time.monotonic() - start_wall > self._budget:
                log.info('train.time_budget_hit', elapsed_sec=int(time.monotonic() - start_wall))
                return
            yield batch

budgeted_loader = _TimeBudgetLoader(loader, TIME_BUDGET_SEC)

# Steps remaining after resume so the loop stops at the configured total.
total_steps = int(cfg['experiment']['steps'])
remaining_steps = max(0, total_steps - RESUME_STEP)

train_config = TrainConfig(
    steps=remaining_steps,
    learning_rate=float(optim_conf.get('learning_rate', 1e-4)),
    weight_decay=float(optim_conf.get('weight_decay', 1e-6)),
    log_every=int(cfg['experiment'].get('log_every', 50)),
    checkpoint_every=int(cfg['experiment'].get('checkpoint_every', 500)),
    checkpoint_dir=CKPT_LIVE,
    device='cuda',
    seed=int(cfg['experiment'].get('seed', 0)),
)
log.info('train.start', total_steps=total_steps, resumed_from=RESUME_STEP,
         steps_this_session=remaining_steps, budget_hours=TIME_BUDGET_SEC / 3600)

if remaining_steps == 0:
    print('Training already complete per config; skipping training call.')
    result = None
else:
    result = train_generator(
        generator=generator, batches=budgeted_loader,
        composite_loss=composite, eot=eot,
        config=train_config, restorer_sampler=restorer_sampler,
    )
    log.info('train.done',
             steps_this_session=len(result.history),
             final_ckpt=str(result.checkpoint_path) if result.checkpoint_path else None,
             final_loss=round(result.history[-1].total_loss, 4) if result.history else None,
             wall_sec=int(time.monotonic() - start_wall))

In [ ]:
# Cell 8 - publish checkpoints as a new Kaggle Dataset version.
#
# Kaggle notebooks are auto-authenticated (kaggle.json is provisioned
# at runtime for the notebook owner). We never write credentials in
# this notebook so it's safe to make public.
#
# First run: creates the dataset. Subsequent runs: `datasets version`.
#
# Set KAGGLE_OWNER to your Kaggle username. If unset the cell reads it
# from the kaggle CLI config that Kaggle mounted for us.
import json, os, subprocess, shutil, glob
from pathlib import Path
from datetime import datetime, timezone

KAGGLE_DATASET_SLUG = 'voidface-training-ckpts'
KAGGLE_OWNER = os.environ.get('KAGGLE_USERNAME') or os.environ.get('KAGGLE_OWNER')

if KAGGLE_OWNER is None:
    # Kaggle mounts creds at /root/.kaggle/kaggle.json or /kaggle/... in
    # newer images. We only read `username`; never print `key`.
    for candidate in ('/root/.kaggle/kaggle.json', '/kaggle/.kaggle/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json')):
        if os.path.exists(candidate):
            with open(candidate) as f:
                KAGGLE_OWNER = json.load(f).get('username')
            break

if not KAGGLE_OWNER:
    print('KAGGLE_OWNER not set and no kaggle.json found. Set KAGGLE_USERNAME env var and rerun this cell, or upload manually from /kaggle/working/checkpoints.')
else:
    # Prune to keep the dataset small - retain last 3 checkpoints only.
    ckpts = sorted(CKPT_LIVE.glob('voidface-step-*.pt'), key=lambda p: p.stat().st_mtime)
    for old in ckpts[:-3]:
        print(f'pruning old ckpt: {old.name}')
        old.unlink()
    ckpts = sorted(CKPT_LIVE.glob('voidface-step-*.pt'), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        print('no checkpoints to publish (training may have stopped before first save)')
    else:
        latest_step = int(ckpts[-1].stem.split('-')[-1])
        meta_path = CKPT_LIVE / 'dataset-metadata.json'
        meta = {
            'title': 'voidface-training-ckpts',
            'id': f'{KAGGLE_OWNER}/{KAGGLE_DATASET_SLUG}',
            'licenses': [{'name': 'CC0-1.0'}],
        }
        meta_path.write_text(json.dumps(meta, indent=2))

        # Decide create-vs-version: dataset exists if `datasets status` succeeds.
        stamp = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
        msg = f'step {latest_step} @ {stamp}'
        status = subprocess.run(
            ['kaggle', 'datasets', 'status', f'{KAGGLE_OWNER}/{KAGGLE_DATASET_SLUG}'],
            capture_output=True, text=True,
        )
        if status.returncode == 0:
            print(f'[publish] versioning existing dataset: {msg}')
            subprocess.run(
                ['kaggle', 'datasets', 'version', '-p', str(CKPT_LIVE),
                 '-m', msg, '--dir-mode', 'zip'],
                check=True,
            )
        else:
            print(f'[publish] creating new dataset {KAGGLE_OWNER}/{KAGGLE_DATASET_SLUG}')
            subprocess.run(
                ['kaggle', 'datasets', 'create', '-p', str(CKPT_LIVE),
                 '--dir-mode', 'zip'],
                check=True,
            )
        print(f'[publish] done. Attach {KAGGLE_OWNER}/{KAGGLE_DATASET_SLUG} to your next session to resume.')

In [ ]:
# Cell 9 - final summary + loss curve.
import time
from pathlib import Path

print('=' * 60)
print('voidface Kaggle session summary')
print('=' * 60)
print(f'Finished (UTC):    {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}')
print(f'Wall this session: {int(time.monotonic() - start_wall)} s')
print(f'Resumed from step: {RESUME_STEP}')
print(f'Total steps goal:  {total_steps}')
if result is not None and result.history:
    print(f'Steps this session:{len(result.history)}')
    print(f'Final total loss:  {result.history[-1].total_loss:.4f}')
    print(f'Final LPIPS:       {result.history[-1].lpips:.4f}')
    print(f'Final restorer:    {result.history[-1].restorer}')
    print(f'Final ckpt path:   {result.checkpoint_path}')
else:
    print('No steps executed this session (training already complete or stopped early).')

ckpt_files = sorted(CKPT_LIVE.glob('voidface-step-*.pt'))
print(f'\nCheckpoints in /kaggle/working/checkpoints/:')
for c in ckpt_files:
    size_mb = c.stat().st_size / 1e6
    print(f'  {c.name}  ({size_mb:.1f} MB)')

if result is not None and result.history:
    try:
        import matplotlib.pyplot as plt
        steps = [h.step + 1 + RESUME_STEP for h in result.history]
        total = [h.total_loss for h in result.history]
        lpips = [h.lpips for h in result.history]
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax1.plot(steps, total, label='total loss', color='#c0392b')
        ax1.set_xlabel('step')
        ax1.set_ylabel('total loss', color='#c0392b')
        ax1.tick_params(axis='y', labelcolor='#c0392b')
        ax2 = ax1.twinx()
        ax2.plot(steps, lpips, label='LPIPS', color='#2c3e50', alpha=0.6)
        ax2.set_ylabel('LPIPS', color='#2c3e50')
        ax2.tick_params(axis='y', labelcolor='#2c3e50')
        fig.suptitle('voidface training (this session)')
        fig.tight_layout()
        plt.show()
    except Exception as exc:  # pragma: no cover - plotting best-effort
        print(f'(plot skipped: {exc})')